In [10]:
import logging
import sys
import os
import pandas as pd
from IPython.display import display, Markdown

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.rag.retriever import ScholarshipRetriever
from src.schemas.student import StudentProfile

logging.info("Initializing Retriever Engine...")
try:
    retriever = ScholarshipRetriever()
    logging.info("Retriever Engine is ready for testing.")
except Exception as e:
    logging.error(f"Failed to initialize Retriever: {e}")
    sys.exit(1)

profile_ai_master = StudentProfile(
    nationality="Syrian",
    academic_level="Master",
    academic_major="Artificial Intelligence",
    gpa=3.8,
    target_countries=["Germany", "USA"],
    skills=["Python", "PyTorch", "NLP", "Machine Learning"],
    age=24
)

profile_cs_bachelor = StudentProfile(
    nationality="Egyptian",
    academic_level="Bachelor",
    academic_major="Computer Science",
    gpa=2.2,
    target_countries=["UK", "Canada"],
    skills=["Java", "C++", "Web Development"],
    age=20
)

profile_unknown = StudentProfile(
    nationality="Unknown",
    academic_level="Bachelor",
    academic_major="Unknown",
    gpa=0.0,
    target_countries=[],
    skills=[],
    age=None
)

TEST_CASES = [
    {
        "name": "Scenario 1: AI Master - Exact Match Retrieval",
        "profile": profile_ai_master,
        "query": "What are the exact application steps and link for the DAAD scholarship?",
        "expected_insight": "Should retrieve DAAD context with steps and URL."
    },
    {
        "name": "Scenario 2: AI Master - Semantic Search via Skills",
        "profile": profile_ai_master,
        "query": "Fully funded scholarships focusing on deep learning and natural language processing.",
        "expected_insight": "Should leverage the skills list to find highly relevant NLP/ML grants."
    },
    {
        "name": "Scenario 3: CS Bachelor - Strict Level Filtering",
        "profile": profile_cs_bachelor,
        "query": "I want to apply for a PhD fellowship or Postdoc position.",
        "expected_insight": "Must reject PhD/Postdoc results and strictly return Bachelor level scholarships."
    },
    {
        "name": "Scenario 4: CS Bachelor - GPA Constraint Filtering",
        "profile": profile_cs_bachelor,
        "query": "Show me highly competitive scholarships.",
        "expected_insight": "Should filter out scholarships requiring GPA > 3.0."
    },
    {
        "name": "Scenario 5: Unknown Profile - Fallback Handling",
        "profile": profile_unknown,
        "query": "Find me a scholarship to study abroad.",
        "expected_insight": "Should return generic Bachelor scholarships without throwing validation errors."
    },
    {
        "name": "Scenario 6: Unknown Profile - Noise and Out of Domain",
        "profile": profile_unknown,
        "query": "What are the latest advancements in quantum computing hardware?",
        "expected_insight": "Should return low similarity scores or handle out-of-domain gracefully."
    }
]

def run_retrieval_diagnostics(test_cases, top_k=3):
    for idx, test in enumerate(test_cases, 1):
        display(Markdown(f"## {test['name']}"))
        logging.info(f"Executing Query: '{test['query']}'")
        logging.info(f"Active Profile: Level={test['profile'].academic_level}, GPA={test['profile'].gpa}")
        logging.info(f"Expected Insight: {test['expected_insight']}")
        
        try:
            retrieved_docs = retriever.match_scholarships(
                profile=test['profile'], 
                query=test['query'], 
                top_k=top_k
            )
            
            if not retrieved_docs:
                logging.warning("No documents retrieved. Check metadata filters or database connection.")
                logging.info("-" * 80)
                continue
                
            results_data = []
            for rank, doc in enumerate(retrieved_docs, 1):
                metadata = doc.metadata
                content_preview = doc.page_content.replace('\n', ' ')
                
                has_link = "http" in content_preview or "www" in content_preview
                has_steps = "step" in content_preview.lower() or "apply" in content_preview.lower()
                
                results_data.append({
                    "Rank": rank,
                    "Scholarship": metadata.get("scholarship_name", "Unknown"),
                    "Level": metadata.get("academic_level", "N/A"),
                    "Score": round(float(metadata.get("rerank_score", getattr(doc, 'score', 0))), 3),
                    "Has Link": "Yes" if has_link else "No",
                    "Has Steps": "Yes" if has_steps else "No",
                    "Snippet": content_preview[:150] + "..."
                })
            
            df = pd.DataFrame(results_data)
            display(df)
            
            top_doc = retrieved_docs[0]
            logging.info(f"Deep Dive - Top Rank Metadata: {top_doc.metadata}")
            logging.debug(f"Deep Dive - Top Rank Content: \n{top_doc.page_content}")
            
        except Exception as e:
            logging.error(f"Error during retrieval execution: {e}", exc_info=True)
            
        logging.info("-" * 80)

run_retrieval_diagnostics(TEST_CASES, top_k=3)

2026-08-15 16:27:30 - INFO - Initializing Retriever Engine...
2026-08-15 16:27:30 - INFO - Initializing Advanced Hybrid Retriever...
2026-08-15 16:27:30 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-08-15 16:27:30 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
2026-08-15 16:27:30 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-08-15 16:27:30 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-08-15 16:27:30 - INFO - Loading SentenceTransformer model from BAAI/bge-m3.
2026-08-15 16:27:30 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/co

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-08-15 16:27:31 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/refs%2Fpr%2F130 "HTTP/1.1 200 OK"
2026-08-15 16:27:31 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/refs%2Fpr%2F130/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-08-15 16:27:31 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-08-15 16:27:31 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/refs%2Fpr%2F130/model.safetensors "HTTP/1.1 302 Found"
2026-08-15 16:27:31 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-08-15 16:27:31 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-08-15 16:27:31 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.j

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-08-15 16:27:42 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-08-15 16:27:42 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-08-15 16:27:42 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-08-15 16:27:42 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-08-15 16:27:42 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-15 16:27:42 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/tokenizer_config.json "HTTP/1.1 200 OK"
2026-0

...

2026-08-15 16:28:23 - WARNING - System retrieved candidates, but ALL failed the 0.500 confidence threshold.
2026-08-15 16:28:23 - WARNING - No documents retrieved. Check metadata filters or database connection.
2026-08-15 16:28:23 - INFO - --------------------------------------------------------------------------------


## Scenario 6: Unknown Profile - Noise and Out of Domain

2026-08-15 16:28:23 - INFO - Executing Query: 'What are the latest advancements in quantum computing hardware?'
2026-08-15 16:28:23 - INFO - Active Profile: Level=Bachelor, GPA=0.0
2026-08-15 16:28:23 - INFO - Expected Insight: Should return low similarity scores or handle out-of-domain gracefully.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-15 16:28:28 - WARNING - System retrieved candidates, but ALL failed the 0.500 confidence threshold.
2026-08-15 16:28:28 - WARNING - No documents retrieved. Check metadata filters or database connection.
2026-08-15 16:28:28 - INFO - --------------------------------------------------------------------------------


In [9]:
import os
import json
import uuid
import logging
import pandas as pd
from IPython.display import display, Markdown
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

def run_retrieval_evaluation():
    logging.info("Initializing Retrieval Evaluation...")
    
    dataset_path = "test_dataset.json"
    if not os.path.exists(dataset_path):
        logging.error("Dataset not found at %s", dataset_path)
        return
        
    with open(dataset_path, "r", encoding="utf-8") as f:
        test_cases = json.load(f)

    logging.info("Loading Embeddings (BAAI/bge-small-en-v1.5)...")
    embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

    logging.info("Injecting 100 documents into ephemeral ChromaDB...")
    docs = []
    
    for idx, case in enumerate(test_cases):
        doc_id = f"real_doc_{idx}"
        case["target_doc_id"] = doc_id
        
        doc = Document(
            page_content=case["contexts"][0] if isinstance(case.get("contexts"), list) else case.get("scholarship_context", ""),
            metadata={"doc_id": doc_id, "type": "real", "case_name": case["name"]}
        )
        docs.append(doc)

    noise_topics = [
        "Italian cooking recipes", 
        "Football match results", 
        "History of the Roman Empire", 
        "Basic car maintenance", 
        "Quantum computing for dummies"
    ]
    for i in range(70):
        noise_content = f"This is a random document about {noise_topics[i % len(noise_topics)]}. It contains noise data to distract the retriever. Unique ID: {uuid.uuid4()}"
        doc = Document(
            page_content=noise_content,
            metadata={"doc_id": f"noise_doc_{i}", "type": "noise"}
        )
        docs.append(doc)

    vector_store = Chroma.from_documents(docs, embeddings)
    logging.info("ChromaDB Indexing Complete.")

    logging.info("Running Queries and Calculating Metrics (K=5)...")
    results = []
    
    for case in test_cases:
        query = case["question"]
        target_id = case["target_doc_id"]
        
        retrieved_docs = vector_store.similarity_search(query, k=5)
        retrieved_ids = [d.metadata["doc_id"] for d in retrieved_docs]
        
        recall_at_5 = 1.0 if target_id in retrieved_ids else 0.0
        
        mrr = 0.0
        if target_id in retrieved_ids:
            rank = retrieved_ids.index(target_id) + 1
            mrr = 1.0 / rank
            
        results.append({
            "Test Case": case["name"],
            "Recall@5": recall_at_5,
            "MRR": mrr
        })

    df = pd.DataFrame(results)
    
    display(Markdown("## Retrieval Evaluation Dashboard (Raw Embedding Benchmark)"))
    display(df)
    
    avg_recall = df['Recall@5'].mean()
    avg_mrr = df['MRR'].mean()
    
    display(Markdown(f"**AVERAGE RECALL@5:** {avg_recall:.2f}"))
    display(Markdown(f"**AVERAGE MRR:** {avg_mrr:.2f}"))

if __name__ == "__main__":
    run_retrieval_evaluation()

2026-08-15 16:26:22 - INFO - Initializing Retrieval Evaluation...
2026-08-15 16:26:22 - INFO - Loading Embeddings (BAAI/bge-small-en-v1.5)...
/tmp/ipykernel_4719/4025061993.py:25: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
2026-08-15 16:26:22 - INFO - No device provided, using cpu
2026-08-15 16:26:22 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-08-15 16:26:22 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-08-15 16:26:22 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-08-15 16:26:22 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-08-15 16:26:22 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-08-15 16:26:23 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-08-15 16:26:23 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-15 16:26:23 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json "HTTP/1.1 200 OK"
2026-0

## Retrieval Evaluation Dashboard (Raw Embedding Benchmark)

,Test Case,Recall@5,MRR
0,Case 1: AI Enthusiast,1.0,0.500000
1,Case 2: Data Science Whiz,1.0,0.500000
2,Case 3: CS Major,1.0,1.000000
3,Case 4: AI Researcher,1.0,1.000000
4,Case 5: Data Engineer,1.0,1.000000
5,Case 6: Machine Learning Expert,1.0,1.000000
6,Case 7: Computer Vision Specialist,1.0,1.000000
7,Case 8: Natural Language Processing Expert,1.0,1.000000
8,Case 9: Robotics Engineer,1.0,1.000000
9,Case 10: Human-Computer Interaction Specialist,1.0,1.000000


**AVERAGE RECALL@5:** 0.90

**AVERAGE MRR:** 0.84